<a href="https://colab.research.google.com/github/alexsiks/gerador-podcast/blob/main/Modelo_de_An%C3%A1lise_IA_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Importar Bibliotecas de Audio

In [6]:
# Instalar o edge-tts para geração de áudio com vozes neurais humanas de alta fidelidade
!pip install -q edge-tts

Importação de Bibliotecas

In [7]:
# ==========================================
# 1. IMPORTAÇÕES
# ==========================================
import asyncio
import edge_tts
from pathlib import Path
from IPython.display import Audio, display

POO do Projeto

In [23]:
class Podcast:
    def __init__(self, roteiro_podcast, dicionario_pronuncias=None):
        # ==========================================
        # 2. CONFIGURAÇÕES DO PODCAST
        # ==========================================
        # Usando uma das vozes brasileiras mais fluidas e estáveis do Microsoft Edge
        self.VOICE = "pt-BR-ThalitaNeural"
        self.OUTPUT_FILE = "podcast_neural.mp3"

        # Ajuste de velocidade leve para deixar a fala natural e dinâmica
        self.RATE = "-2%"
        self.VOLUME = "+0%"
        self.PITCH = "+0Hz"
        self.VOICE_FALLBACK = "pt-BR-AntonioNeural"
        self.roteiro_podcast = roteiro_podcast

        # Dicionário de pronúncias personalizadas (ex: {"íri": "íris", "AI": "A.I."})
        self.dicionario_pronuncias = dicionario_pronuncias if dicionario_pronuncias is not None else {}

    # ==========================================
    # 3. AJUSTE DE PRONÚNCIA
    # ==========================================
    def aplicar_pronuncias(self):
        """Substitui palavras específicas no texto para melhorar a pronúncia."""
        texto_ajustado = self.roteiro_podcast
        if not texto_ajustado:
            return texto_ajustado

        for palavra, pronuncia_correta in self.dicionario_pronuncias.items():
            # Faz a substituição ignorando maiúsculas/minúsculas de forma simples
            import re
            padrao = re.compile(re.escape(palavra), re.IGNORECASE)
            texto_ajustado = padrao.sub(pronuncia_correta, texto_ajustado)

        return texto_ajustado

    # ==========================================
    # 4. FUNÇÃO DE GERAÇÃO DO ÁUDIO
    # ==========================================
    async def gerar_podcast(self):
        if not self.roteiro_podcast:
            raise ValueError("O roteiro do podcast não pode estar vazio!")

        # Aplica o mapeamento de pronúncias antes de enviar ao sintetizador
        texto_final = self.aplicar_pronuncias()
        print(f"Texto original: \"{self.roteiro_podcast}\"")
        if self.dicionario_pronuncias:
            print(f"Texto com ajustes de pronúncia: \"{texto_final}\"")

        try:
            comunicacao = edge_tts.Communicate(
                text=texto_final,
                voice=self.VOICE,
                rate=self.RATE,
                volume=self.VOLUME,
                pitch=self.PITCH
            )
            await comunicacao.save(self.OUTPUT_FILE)
            return self.OUTPUT_FILE
        except Exception as e:
            print(f"Erro na síntese com a voz {self.VOICE}: {e}")
            # Fallback para voz masculina caso ocorra falha de rede ou de voz específica

            print(f"Tentando voz de segurança: {self.VOICE_FALLBACK}...")
            comunicacao = edge_tts.Communicate(
                text=texto_final,
                voice=self.VOICE_FALLBACK,
                rate=self.RATE,
                volume=self.VOLUME,
                pitch=self.PITCH
            )
            await comunicacao.save(self.OUTPUT_FILE)
            return self.OUTPUT_FILE

    # ==========================================
    # 5. EXECUÇÃO E REPRODUÇÃO
    # ==========================================
    async def executar(self):
        print("Iniciando a síntese de voz neural ultra-realista...")
        print(f"Voz principal: {self.VOICE}")

        arquivo_gerado = await self.gerar_podcast()
        caminho = Path(arquivo_gerado)

        if caminho.exists() and caminho.stat().st_size > 0:
            tamanho_kb = caminho.stat().st_size / 1024
            print("\n🎙️ Podcast gerado com absoluto sucesso!")
            print(f"Arquivo salvo em: {caminho.name}")
            print(f"Tamanho do áudio: {tamanho_kb:.2f} KB\n")

            display(
                Audio(
                    filename=str(caminho),
                    autoplay=True
                )
            )
        else:
            print("Erro crítico: O arquivo não pôde ser gerado.")

Realizar a geração do Audio

In [42]:
# Mapeamento para corrigir pronúncias de palavras específicas
mapeamento_pronuncias = {
    "space x": "space éxi"
}

podcast = Podcast("space x", dicionario_pronuncias=mapeamento_pronuncias)

# Executa o podcast com a pronúncia corrigida
await podcast.executar()

Iniciando a síntese de voz neural ultra-realista...
Voz principal: pt-BR-ThalitaNeural
Texto original: "space x"
Texto com ajustes de pronúncia: "space éxi"

🎙️ Podcast gerado com absoluto sucesso!
Arquivo salvo em: podcast_neural.mp3
Tamanho do áudio: 10.97 KB

